In [0]:
# Databricks notebook source
# CELL 1: Discovery Configuration, Exclusions & Setup
import re
from sodapy import Socrata
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import current_timestamp, col, max as spark_max
from pyspark.sql import functions as F
from datetime import datetime

# Widgets
dbutils.widgets.text("socrata_domain", "opendata.maryland.gov", "Socrata Domain")
dbutils.widgets.text("catalog_name", "", "Catalog Name")
dbutils.widgets.text("schema_name", "meta", "Control Schema Name")
dbutils.widgets.text("table_name", "_socrata_ingestion_queue", "Control Table Name")
dbutils.widgets.text("limit", "", "Dataset Limit (Leave blank for ALL)")
dbutils.widgets.dropdown("force_refresh", "false", ["true", "false"], "Force Full Re-queue (Ignore Historical Watermarks)")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")

SOCRATA_DOMAIN = dbutils.widgets.get("socrata_domain")
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
TABLE_NAME = dbutils.widgets.get("table_name")
FORCE_REFRESH = dbutils.widgets.get("force_refresh").lower() == "true"
VERBOSE = dbutils.widgets.get("verbose").lower() == "true"

limit_str = dbutils.widgets.get("limit").strip()
DATASET_LIMIT = int(limit_str) if limit_str and limit_str not in ["0", "-1"] else None

QUEUE_TABLE = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{TABLE_NAME}`"

RULES_TABLE = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`_socrata_ingestion_rules`"

def log(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)

def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', (name or '').lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:128]
# Before dispatching tasks to the queue or passing them to the for_each job:
def filter_unhealthy_datasets(spark, queue_table: str, candidate_ids: list) -> set:
    """
    Returns candidate IDs that are NOT in a cooldown/quarantined state.
    """
    if not candidate_ids:
        return set()

    formatted_ids = ",".join([f"'{cid}'" for cid in candidate_ids])

    # Datasets quarantined OR failed within the last 24 hours
    blocked_df = spark.sql(f"""
        WITH latest_log AS (
            SELECT 
                dataset_id,
                status,
                logged_at,
                ROW_NUMBER() OVER (PARTITION BY dataset_id ORDER BY logged_at DESC) as rn
            FROM {queue_table}
            WHERE dataset_id IN ({formatted_ids})
              AND task_type = 'DATA'
        )
        SELECT dataset_id
        FROM latest_log
        WHERE rn = 1
          AND (
              status = 'QUARANTINED'
              OR (status = 'FAILED' AND logged_at >= current_timestamp() - INTERVAL 72 HOURS)
          )
    """).collect()

    blocked_set = {row["dataset_id"] for row in blocked_df}
    return set(candidate_ids) - blocked_set


In [0]:
# COMMAND ----------
# CELL 2: Read Historical Watermarks & Discover Changed Datasets
def load_governance_rules():
    """
    Loads active exclusion rules from the Unity Catalog metadata schema.
    Falls back gracefully if the table is empty or uninitialized.
    """
    excluded_ids = set()
    excluded_keywords = []
    excluded_domains = []

    clean_rules_table = RULES_TABLE.replace("`", "")
    if spark.catalog.tableExists(clean_rules_table):
        try:
            rules_df = spark.table(RULES_TABLE).filter("is_active = true AND action = 'EXCLUDE'")
            for row in rules_df.collect():
                rtype = (row["rule_type"] or "").strip().lower()
                rval = (row["rule_value"] or "").strip()

                if not rval:
                    continue

                if rtype == "dataset_id":
                    excluded_ids.add(rval.lower())
                elif rtype == "title_keyword":
                    excluded_keywords.append(rval.lower())
                elif rtype == "attribution_domain":
                    excluded_domains.append(rval.lower())

            log(f"Loaded governance rules: {len(excluded_ids)} IDs, {len(excluded_keywords)} keywords, {len(excluded_domains)} domains.")
        except Exception as e:
            log(f"Warning: Failed to load governance rules from table: {e}. Running without external rules.")
    else:
        log(f"Notice: Rules table {RULES_TABLE} not found. Running with no external exclusions.")

    return excluded_ids, excluded_keywords, excluded_domains

# Load rules dynamically
EXCLUDED_DATASET_IDS, EXCLUDED_TITLE_KEYWORDS, EXCLUDED_ATTRIBUTION_DOMAINS = load_governance_rules()

SECRET_SCOPE = "socrata_scope"
try:
    APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
    log("Successfully retrieved Socrata app token from secret scope.")
except Exception:
    APP_TOKEN = None
    log("Warning: Running unauthenticated. Requests will be subject to throttling.")

client = Socrata(SOCRATA_DOMAIN, APP_TOKEN, timeout=60)

# 1. Read Historical Watermarks Separately for DATA and METADATA
existing_data_watermarks = {}
existing_metadata_watermarks = {}
previously_known_datasets = {}
table_exists = spark.catalog.tableExists(QUEUE_TABLE.replace("`", ""))

if table_exists and not FORCE_REFRESH:
    try:
        prev_df = spark.table(QUEUE_TABLE)
        
        # Load all datasets previously ingested that are not currently deprecated
        latest_status = (
            prev_df.groupBy("dataset_id")
                   .agg(
                       F.last("status").alias("current_status"),
                       F.last("table_name").alias("table_name")
                   )
                   .filter("current_status != 'DEPRECATED'")
                   .collect()
        )
        previously_known_datasets = {r["dataset_id"]: r["table_name"] for r in latest_status}

        # Data watermarks
        data_col = "portal_data_updated_at"
        data_cond = f"status = 'SUCCESS' AND {data_col} IS NOT NULL"
        if "task_type" in prev_df.columns:
            data_cond += " AND (task_type = 'DATA' OR task_type IS NULL)"
            
        data_rows = (prev_df.filter(data_cond)
                            .groupBy("dataset_id")
                            .agg(spark_max(data_col).alias("max_data_update"))
                            .collect())
        existing_data_watermarks = {r["dataset_id"]: str(r["max_data_update"]).strip() for r in data_rows}

        # Metadata watermarks
        meta_col = "portal_metadata_updated_at"
        meta_cond = f"status = 'SUCCESS' AND {meta_col} IS NOT NULL"
        if "task_type" in prev_df.columns:
            meta_cond += " AND (task_type = 'METADATA' OR task_type IS NULL)"
            
        meta_rows = (prev_df.filter(meta_cond)
                            .groupBy("dataset_id")
                            .agg(spark_max(meta_col).alias("max_meta_update"))
                            .collect())
        existing_metadata_watermarks = {r["dataset_id"]: str(r["max_meta_update"]).strip() for r in meta_rows}

        log(f"Loaded {len(existing_data_watermarks)} data watermarks and {len(existing_metadata_watermarks)} metadata watermarks.")
    except Exception as e:
        log(f"Warning: Could not read prior watermarks: {e}. Defaulting to full discovery.")

def discover_socrata_catalog(domain: str, max_limit: int = None) -> tuple:
    log(f"Querying Socrata discovery catalog for {domain} via sodapy...")
    try:
        catalog_results = client.datasets(limit=0)
    except Exception as err:
        raise RuntimeError(f"Failed to query Socrata Discovery API: {err}")

    log(f"Total raw items returned from catalog: {len(catalog_results)}")

    discovered = []
    seen_valid_ids = set()
    excluded_from_pipeline_ids = set()
    
    skipped_unchanged = 0
    excluded_count = 0
    skipped_mdgeodata = 0
    skipped_non_public = 0
    skipped_non_tabular = 0

    existing_bronze_tables = set(
    row.tableName.lower() 
    for row in spark.sql(f"SHOW TABLES IN `{CATALOG_NAME}`.bronze").collect()
    )

    for item in catalog_results:
        resource = item.get("resource", {})
        classification = item.get("classification", {})
        dataset_id = (resource.get("id") or "").strip()
        title = (resource.get("name") or dataset_id).strip()
        description = (resource.get("description") or "").lower()

        if not dataset_id:
            continue

        # Exclusion Rule 1: Exact ID (normalized case and trimmed)
        if dataset_id.lower() in EXCLUDED_DATASET_IDS:
            excluded_count += 1
            excluded_from_pipeline_ids.add(dataset_id)
            continue

        # Exclusion Rule 2: Title and Description keywords
        title_lower = title.lower()
        if any(keyword in title_lower or keyword in description for keyword in EXCLUDED_TITLE_KEYWORDS):
            excluded_count += 1
            excluded_from_pipeline_ids.add(dataset_id)
            continue

        # Exclusion Rule 3: Source attribution domain (mdgeodata)
        attr_link = str(resource.get("attribution_link") or "").lower()
        attr_text = str(resource.get("attribution") or "").lower()
        if any(dom in attr_link or dom in attr_text for dom in EXCLUDED_ATTRIBUTION_DOMAINS):
            skipped_mdgeodata += 1
            excluded_from_pipeline_ids.add(dataset_id)
            continue

        # Inclusion Rule 1: Official Tabular Datasets only
        resource_type = str(resource.get("type") or "").strip().lower()
        asset_type = str(item.get("type") or "").strip().lower()
        is_tabular = (resource_type == "dataset" or asset_type == "dataset")
        provenance = str(resource.get("provenance") or "").strip().lower() == "official"

        if not (is_tabular and provenance):
            skipped_non_tabular += 1
            excluded_from_pipeline_ids.add(dataset_id)
            continue

        # Inclusion Rule 2: Public Dataset Validation
        approvals = resource.get("approvals") or item.get("approvals") or [{}]
        first_approval = approvals[0] if isinstance(approvals, list) and approvals else {}
        
        target_audience = str(
            first_approval.get("targetAudience") or 
            resource.get("targetAudience") or 
            item.get("targetAudience") or 
            ""
        ).strip().lower()
        
        publication_stage = str(
            resource.get("publicationStage") or 
            item.get("publicationStage") or 
            ""
        ).strip().lower()

        if publication_stage and publication_stage != "published":
            skipped_non_public += 1
            continue
        if target_audience and target_audience != "public":
            skipped_non_public += 1
            continue

        # Active, public dataset confirmed
        seen_valid_ids.add(dataset_id)

        t_name = sanitize_table_name(title)
        portal_data_updated_at = resource.get("data_updated_at")
        portal_metadata_updated_at = resource.get("metadata_updated_at")

        # 1. Determine Data Sync Requirement
        prev_data_update = existing_data_watermarks.get(dataset_id)
        needs_data_sync = (
            FORCE_REFRESH 
            or (dataset_id not in existing_data_watermarks) 
            or (portal_data_updated_at and portal_data_updated_at > prev_data_update)
        )

        # 2. Check if the table exists in Databricks Bronze storage
        table_exists_in_bronze = t_name.lower() in existing_bronze_tables

        # 3. Determine Metadata Sync Requirement
        prev_meta_update = existing_metadata_watermarks.get(dataset_id)
        metadata_timestamp_newer = (
            FORCE_REFRESH 
            or (dataset_id not in existing_metadata_watermarks) 
            or (portal_metadata_updated_at and portal_metadata_updated_at > prev_meta_update)
        )

        # CRITICAL FILTER:
        # Never queue METADATA if the bronze table does not exist, unless DATA is being queued right now
        needs_metadata_sync = metadata_timestamp_newer and (table_exists_in_bronze or needs_data_sync)

        # Skip if neither data nor metadata changed
        if not needs_data_sync and not needs_metadata_sync:
            skipped_unchanged += 1
            continue

        category = classification.get("categories", ["Uncategorized"])
        category_str = category[0] if isinstance(category, list) and category else "Uncategorized"

        # Queue DATA task ONLY if data actually changed
        if needs_data_sync:
            discovered.append({
                "dataset_id": dataset_id,
                "task_type": "DATA",
                "title": title,
                "table_name": t_name,
                "category": category_str,
                "portal_data_updated_at": portal_data_updated_at,
                "portal_metadata_updated_at": None,
                "status": "QUEUED",
                "records_ingested": None,
                "error_message": None
            })

        # Queue METADATA task ONLY if metadata changed AND target table exists / is queued
        if needs_metadata_sync:
            discovered.append({
                "dataset_id": dataset_id,
                "task_type": "METADATA",
                "title": title,
                "table_name": t_name,
                "category": category_str,
                "portal_data_updated_at": None,
                "portal_metadata_updated_at": portal_metadata_updated_at,
                "status": "QUEUED",
                "records_ingested": None,
                "error_message": None
            })
            
        if max_limit and len(discovered) >= max_limit:
            log(f"Reached limit cutoff: {max_limit}")
            break

    log(f"Filtered out {skipped_non_tabular} non-tabular or unofficial items.")
    log(f"Filtered out {skipped_non_public} non-public or unpublished datasets.")
    log(f"Filtered out {excluded_count} datasets by ID/title/tag exclusions.")
    log(f"Filtered out {skipped_mdgeodata} datasets matching excluded attribution (mdgeodata).")
    log(f"Skipped {skipped_unchanged} datasets not updated since last successful ingest.")
    
    # True deprecations = datasets in history that are NOT active AND were NOT intentionally filtered out
    deprecated_ids = (set(previously_known_datasets.keys()) - seen_valid_ids) - excluded_from_pipeline_ids

    return discovered, deprecated_ids

datasets, deprecated_ids = discover_socrata_catalog(SOCRATA_DOMAIN, max_limit=DATASET_LIMIT)

# 1. Extract IDs from the discovered tasks
active_candidate_ids = [d["dataset_id"] for d in datasets]

# 2. Check for quarantine or cooldown
eligible_ids = filter_unhealthy_datasets(spark, QUEUE_TABLE, active_candidate_ids)

# 3. Filter DATA tasks (and keep metadata aligned or drop tasks for unhealthy datasets)
initial_task_count = len(datasets)
datasets = [d for d in datasets if d["dataset_id"] in eligible_ids]

log(f"Discovery Summary: Discovered {initial_task_count} tasks | Blocked/Cooldown: {initial_task_count - len(datasets)} | Final Eligible Tasks: {len(datasets)}")


In [0]:
# COMMAND ----------
# CELL 3: Append Discovered Tasks and Log Deprecations to Queue
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime

schema = StructType([
    StructField("dataset_id", StringType(), False),
    StructField("task_type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("portal_data_updated_at", StringType(), True),
    StructField("portal_metadata_updated_at", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_ingested", LongType(), True),
    StructField("error_message", StringType(), True)
])

# 1. Append Newly Queued Ingestion Tasks
if datasets:
    new_entries_df = (spark.createDataFrame(datasets, schema=schema)
                      .withColumn("logged_at", current_timestamp()))

    (new_entries_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(QUEUE_TABLE))

    log(f"Successfully appended {len(datasets)} task records into audit log: {QUEUE_TABLE}")
else:
    log("All datasets are up-to-date with portal. No new ingestion tasks logged.")

# 2. Append Deprecated Events & Tag Tables in Unity Catalog
if 'deprecated_ids' in locals() and deprecated_ids and not FORCE_REFRESH:
    deprecated_rows = []
    dep_time = datetime.utcnow().isoformat()
    
    for dep_id in deprecated_ids:
        t_name = previously_known_datasets.get(dep_id, dep_id)
        deprecated_rows.append({
            "dataset_id": dep_id,
            "task_type": "DEPRECATED",
            "title": t_name,
            "table_name": t_name,
            "category": "Deprecated",
            "portal_data_updated_at": None,
            "portal_metadata_updated_at": None,
            "status": "DEPRECATED",
            "records_ingested": 0,
            "error_message": f"Privated or deleted from open data portal as of {dep_time}"
        })
        
        # Tag both Bronze and Silver tables in Unity Catalog if they exist
        for target_schema in ["bronze", "silver"]:
            full_table = f"`{CATALOG_NAME}`.`{target_schema}`.`{t_name}`"
            if spark.catalog.tableExists(full_table.replace("`", "")):
                try:
                    spark.sql(f"ALTER TABLE {full_table} SET TAGS ('lifecycle' = 'deprecated')")
                    spark.sql(f"""
                        COMMENT ON TABLE {full_table} IS 
                        'DEPRECATED: Upstream dataset was privated or deleted on {dep_time}. Retained for historical continuity.'
                    """)
                except Exception as tag_err:
                    log(f"Warning: Could not tag {full_table}: {tag_err}")

    if deprecated_rows:
        dep_df = (spark.createDataFrame(deprecated_rows, schema=schema)
                       .withColumn("logged_at", current_timestamp()))
        
        (dep_df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(QUEUE_TABLE))
            
        log(f"Successfully appended {len(deprecated_rows)} DEPRECATED events into audit log.")
else:
    log("No deprecated datasets detected.")